# 🧩 Mini-Lab: Chain of Thought Reasoning

**Module 4: Prompt Engineering & Security** | **Duration: ~30 min** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** what chain-of-thought (CoT) prompting is and why it improves reasoning accuracy
2. **Compare** direct (no-CoT) answers vs. chain-of-thought answers on reasoning tasks
3. **Apply** self-consistency by sampling multiple reasoning paths and selecting the most common answer

## Target Concepts

| Concept | Description |
|---------|-------------|
| Chain-of-Thought | Prompting the model to show step-by-step reasoning before giving a final answer, improving accuracy on complex tasks |
| Self-Consistency | Generating multiple CoT reasoning paths and taking a majority vote to improve reliability |

## Setup

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()  # Uses OPENAI_API_KEY from .env

MODEL = "gpt-4o-mini"

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

def chat(user_message, system_message="You are a helpful assistant.", temperature=0):
    """Send a chat completion request and return the response text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

## 1. The Problem: LLMs Rush to Answers

When asked a multi-step reasoning question, LLMs often jump straight to an answer without working through the logic. This leads to mistakes — especially on math, logic, and word problems.

Let's see this in action with a classic reasoning problem.

In [3]:
# A reasoning problem that trips up direct answers
problem = """
A farmer has 3 fields. Each field has 4 rows of crops.
Each row has 8 plants. He sells half of all his plants
and then buys 20 more. How many plants does he have now?
"""

# Direct prompting — just ask for the answer
direct_prompt = f"{problem.strip()}\n\nAnswer with just the final number."

direct_answer = chat(direct_prompt)
md(f"**Direct answer:** {direct_answer}")

**Direct answer:** The farmer has 52 plants now.

The model may get this right or wrong. But even when correct, we can't verify its reasoning. Let's fix that.

## 2. Chain-of-Thought Prompting

**Chain-of-thought (CoT)** prompting asks the model to "think step by step" before giving a final answer. This simple instruction dramatically improves reasoning accuracy.

The key phrase: **"Let's think step by step."**

In [4]:
# Chain-of-thought prompting — ask for step-by-step reasoning
cot_prompt = f"{problem.strip()}\n\nLet's think step by step."

cot_answer = chat(cot_prompt)
md(f"**Chain-of-Thought answer:**\n\n{cot_answer}")

**Chain-of-Thought answer:**

Let's break down the problem step by step.

1. **Calculate the total number of plants in one field:**
   - Each field has 4 rows of crops.
   - Each row has 8 plants.
   - Therefore, the total number of plants in one field is:
     \[
     4 \text{ rows} \times 8 \text{ plants/row} = 32 \text{ plants}
     \]

2. **Calculate the total number of plants in all three fields:**
   - The farmer has 3 fields.
   - Therefore, the total number of plants in all fields is:
     \[
     3 \text{ fields} \times 32 \text{ plants/field} = 96 \text{ plants}
     \]

3. **Calculate how many plants he sells:**
   - The farmer sells half of all his plants.
   - Therefore, the number of plants he sells is:
     \[
     \frac{96 \text{ plants}}{2} = 48 \text{ plants}
     \]

4. **Calculate how many plants he has left after selling:**
   - After selling 48 plants, the number of plants he has left is:
     \[
     96 \text{ plants} - 48 \text{ plants} = 48 \text{ plants}
     \]

5. **Calculate the total number of plants after buying 20 more:**
   - The farmer then buys 20 more plants.
   - Therefore, the total number of plants he has now is:
     \[
     48 \text{ plants} + 20 \text{ plants} = 68 \text{ plants}
     \]

So, the farmer has **68 plants** now.

Notice how the model now shows its work:
- 3 fields × 4 rows × 8 plants = 96 total
- Sells half: 96 / 2 = 48
- Buys 20 more: 48 + 20 = **68**

The step-by-step reasoning makes the answer verifiable and more likely correct.

## 3. Comparing Direct vs. CoT on a Harder Problem

Let's try a trickier logic problem to see the difference more clearly.

In [5]:
logic_problem = """
A store had 125 books. On Monday, they sold 28 books. On Tuesday,
they received a shipment of 45 books but returned 12 defective ones.
On Wednesday, they sold twice as many as Monday. How many books
does the store have now?
"""

# Direct
direct = chat(f"{logic_problem.strip()}\n\nAnswer with just the final number.")

# CoT
cot = chat(f"{logic_problem.strip()}\n\nLet's think step by step.")

md(f"**Direct answer:** {direct}")
print()
md(f"**CoT answer:**\n\n{cot}")

**Direct answer:** The store has 130 books now.

**CoT answer:**

Let's break down the problem step by step.

1. **Initial number of books**: The store starts with 125 books.

2. **Books sold on Monday**: The store sold 28 books on Monday.
   \[
   125 - 28 = 97 \text{ books remaining}
   \]

3. **Books received and returned on Tuesday**: On Tuesday, the store received 45 books but returned 12 defective ones.
   - First, calculate the net number of books received:
   \[
   45 - 12 = 33 \text{ books added}
   \]
   - Now, add these to the remaining books from Monday:
   \[
   97 + 33 = 130 \text{ books remaining}
   \]

4. **Books sold on Wednesday**: On Wednesday, the store sold twice as many books as they sold on Monday. Since they sold 28 books on Monday, they sold:
   \[
   2 \times 28 = 56 \text{ books on Wednesday}
   \]
   - Now, subtract the books sold on Wednesday from the total remaining:
   \[
   130 - 56 = 74 \text{ books remaining}
   \]

So, the store now has **74 books**.

The correct answer is: 125 - 28 + 45 - 12 - 56 = **74**. Check which approach got it right!

## 4. Self-Consistency: Majority Vote Over Multiple Reasoning Paths

Even with CoT, the model might occasionally reason incorrectly. **Self-consistency** improves reliability by:

1. Generating **multiple** CoT responses (with temperature > 0 for diversity)
2. Extracting the final answer from each
3. Taking the **majority vote**

This works because different reasoning paths may make different errors, but the correct answer tends to appear most often.

In [6]:
tricky_problem = """
If a shirt costs $25 and is on sale for 20% off, and you also
have a coupon for $3 off the sale price, and then you need to
pay 8% sales tax on the final price, how much do you pay?
"""

cot_system = "You are a careful math tutor. Always show your step-by-step work, then give your final answer as 'FINAL ANSWER: $X.XX'"

# Generate 5 reasoning paths with temperature=0.7 for diversity
num_samples = 5
responses = []

for i in range(num_samples):
    resp = chat(
        f"{tricky_problem.strip()}\n\nLet's think step by step.",
        system_message=cot_system,
        temperature=0.7
    )
    responses.append(resp)
    print(f"Sample {i+1} collected.")

print(f"\nCollected {num_samples} reasoning paths.")

Sample 1 collected.
Sample 2 collected.
Sample 3 collected.
Sample 4 collected.
Sample 5 collected.

Collected 5 reasoning paths.


In [12]:
# Let's look at each reasoning path
for i, resp in enumerate(responses):
    md(f"### Path {i+1}")
    md(resp)
    print("---")

### Path 1

Sure! Let's break this down step by step.

1. **Calculate the sale price after the discount**:
   The original price of the shirt is $25, and it is on sale for 20% off.

   First, we find 20% of $25:
   \[
   20\% \text{ of } 25 = 0.20 \times 25 = 5
   \]

   Now, subtract the discount from the original price:
   \[
   \text{Sale Price} = 25 - 5 = 20
   \]

2. **Apply the coupon for additional savings**:
   You have a coupon for $3 off the sale price of $20.
   \[
   \text{Final Price after Coupon} = 20 - 3 = 17
   \]

3. **Calculate the sales tax**:
   Now, we need to apply the 8% sales tax on the final price of $17.
   
   First, find 8% of $17:
   \[
   8\% \text{ of } 17 = 0.08 \times 17 = 1.36
   \]

   Now, add the sales tax to the final price:
   \[
   \text{Total Price} = 17 + 1.36 = 18.36
   \]

So, the total amount you pay after the discount, the coupon, and including sales tax is:
\[
\text{FINAL ANSWER: } 18.36
\]

---


### Path 2

Let's break this problem down step by step.

1. **Calculate the discount on the shirt**:
   The original price of the shirt is $25, and it is on sale for 20% off. 

   \[
   \text{Discount} = \text{Original Price} \times \text{Discount Rate}
   \]
   \[
   \text{Discount} = 25 \times 0.20 = 5
   \]

   So, the discount amount is $5.

2. **Calculate the sale price after the discount**:
   To find the sale price, we subtract the discount from the original price.

   \[
   \text{Sale Price} = \text{Original Price} - \text{Discount}
   \]
   \[
   \text{Sale Price} = 25 - 5 = 20
   \]

   The sale price of the shirt is $20.

3. **Apply the coupon**:
   You have a coupon for $3 off the sale price. We need to subtract this coupon amount from the sale price.

   \[
   \text{Final Price Before Tax} = \text{Sale Price} - \text{Coupon}
   \]
   \[
   \text{Final Price Before Tax} = 20 - 3 = 17
   \]

   The final price before tax is $17.

4. **Calculate the sales tax**:
   Now, we need to calculate the 8% sales tax on the final price.

   \[
   \text{Sales Tax} = \text{Final Price Before Tax} \times \text{Sales Tax Rate}
   \]
   \[
   \text{Sales Tax} = 17 \times 0.08 = 1.36
   \]

   The sales tax is $1.36.

5. **Calculate the total amount to pay**:
   Finally, we add the sales tax to the final price before tax to find the total amount you need to pay.

   \[
   \text{Total Amount} = \text{Final Price Before Tax} + \text{Sales Tax}
   \]
   \[
   \text{Total Amount} = 17 + 1.36 = 18.36
   \]

The total amount you need to pay is $18.36.

Thus, the final answer is: 

FINAL ANSWER: $18.36

...

Sure! Let's break this down step by step.

1. **Calculate the sale price of the shirt:**
   The original price of the shirt is $25, and it is on sale for 20% off.
   To find the discount amount, we calculate 20% of $25.

   \[
   \text{Discount} = 0.20 \times 25 = 5
   \]

   Now, subtract the discount from the original price to find the sale price.

   \[
   \text{Sale Price} = 25 - 5 = 20
   \]

2. **Apply the $3 coupon to the sale price:**
   Now we take the sale price of $20 and apply the $3 coupon.

   \[
   \text{Final Price after Coupon} = 20 - 3 = 17
   \]

3. **Calculate the sales tax:**
   Next, we need to calculate the sales tax on the final price of $17. The sales tax rate is 8%.

   \[
   \text{Sales Tax} = 0.08 \times 17 = 1.36
   \]

4. **Calculate the total amount paid:**
   Finally, add the sales tax to the final price after the coupon.

   \[
   \text{Total Amount Paid} = 17 + 1.36 = 18.36
   \]

So, the total amount you pay is 

FINAL ANSWER: $18.36

---


### Path 5

Let's break down the problem step by step.

1. **Calculate the sale price of the shirt after the discount.**
   - The original price of the shirt is $25.
   - The discount is 20% of the original price.
   - First, we calculate 20% of $25:
     \[
     20\% \text{ of } 25 = 0.20 \times 25 = 5
     \]
   - Now, subtract the discount from the original price:
     \[
     \text{Sale price} = \text{Original price} - \text{Discount} = 25 - 5 = 20
     \]

2. **Apply the coupon discount.**
   - You have a coupon for $3 off the sale price.
   - Subtract the coupon value from the sale price:
     \[
     \text{Final price before tax} = \text{Sale price} - \text{Coupon} = 20 - 3 = 17
     \]

3. **Calculate the sales tax on the final price.**
   - The sales tax rate is 8%.
   - First, we calculate 8% of the final price:
     \[
     8\% \text{ of } 17 = 0.08 \times 17 = 1.36
     \]
   - Now, add the sales tax to the final price:
     \[
     \text{Total price} = \text{Final price before tax} + \text{Sales tax} = 17 + 1.36 = 18.36
     \]

So, the total amount you pay after all discounts and taxes is:

FINAL ANSWER: $18.36

---


## 5. Extracting and Voting on Answers

Now let's extract the final answer from each path and take a majority vote.

In [15]:
import re
from collections import Counter

def extract_final_answer(text):
    """Extract dollar amount after 'FINAL ANSWER:' from a CoT response."""
    match = re.search(r'FINAL ANSWER:\s*\}?\s*\$?([\d.]+)', text)
    if match:
        return match.group(1)
    return None

# Extract answers from all paths
answers = []
for i, resp in enumerate(responses):
    answer = extract_final_answer(resp)
    answers.append(answer)
    print(f"Path {i+1}: ${answer}")

# Majority vote
valid_answers = [a for a in answers if a is not None]
vote_counts = Counter(valid_answers)

print(f"\nVote counts: {dict(vote_counts)}")

winner = vote_counts.most_common(1)[0]
md(f"**🏆 Self-consistent answer: ${winner[0]}** (appeared {winner[1]}/{num_samples} times)")

Path 1: $18.36
Path 2: $18.36
Path 3: $18.36
Path 4: $18.36
Path 5: $18.36

Vote counts: {'18.36': 5}


**🏆 Self-consistent answer: $18.36** (appeared 5/5 times)

The correct answer: $25 × 0.80 = $20.00, minus $3 = $17.00, plus 8% tax = **$18.36**.

Self-consistency helps catch the occasional wrong reasoning path by relying on the consensus.

## 6. When to Use CoT vs. Self-Consistency

| Technique | Best For | Cost | Reliability |
|-----------|----------|------|-------------|
| **Direct** | Simple factual questions | 1 call | Baseline |
| **CoT** | Multi-step reasoning, math, logic | 1 call (more tokens) | Better |
| **Self-Consistency** | High-stakes reasoning where accuracy matters most | N calls | Best |

**Rules of thumb:**
- Use **direct prompting** for simple lookups and classification
- Use **CoT** when the task requires multiple reasoning steps
- Use **self-consistency** when you need high confidence and can afford extra API calls

## 🎯 Summary

### Key Takeaways

1. **Chain-of-thought prompting** — adding "Let's think step by step" makes the model show its reasoning, dramatically improving accuracy on multi-step problems
2. **Verifiable reasoning** — CoT outputs let you inspect and verify the model's logic, not just its final answer
3. **Self-consistency** — sampling multiple CoT paths and taking a majority vote increases reliability at the cost of additional API calls
4. **Right tool for the job** — use direct prompting for simple tasks, CoT for reasoning tasks, and self-consistency when accuracy is critical